In [138]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder 
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
import xgboost as xgb
from xgboost.sklearn import XGBClassifier
from xgboost.sklearn import XGBRegressor

In [139]:
df=pd.read_csv(r'E:\code\AI study\AI study\ML\code\5) ML Model\8) Ensemble\6) XGBoost\city_hour.csv')

In [140]:
df.head()

,City,Datetime,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-01 01:00:00,NaN,NaN,1.00,40.01,36.37,NaN,1.00,122.07,NaN,0.0,0.0,0.0,NaN,NaN
1,Ahmedabad,2015-01-01 02:00:00,NaN,NaN,0.02,27.75,19.73,NaN,0.02,85.90,NaN,0.0,0.0,0.0,NaN,NaN
2,Ahmedabad,2015-01-01 03:00:00,NaN,NaN,0.08,19.32,11.08,NaN,0.08,52.83,NaN,0.0,0.0,0.0,NaN,NaN
3,Ahmedabad,2015-01-01 04:00:00,NaN,NaN,0.30,16.45,9.20,NaN,0.30,39.53,153.58,0.0,0.0,0.0,NaN,NaN
4,Ahmedabad,2015-01-01 05:00:00,NaN,NaN,0.12,14.90,7.85,NaN,0.12,32.63,NaN,0.0,0.0,0.0,NaN,NaN


In [141]:
df.describe()

,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI
count,562787.000000,411138.000000,591243.000000,590753.000000,584651.000000,435333.000000,621358.000000,577502.000000,578667.000000,544229.000000,487268.000000,252046.000000,578795.000000
mean,67.622994,119.075804,17.421755,28.885157,32.287565,23.607959,2.183539,14.038307,34.798979,3.087595,8.660927,3.130537,166.413500
std,74.730496,104.224752,32.095211,29.162194,39.756669,28.831900,10.970514,19.305540,29.806379,16.456599,21.741023,7.834832,162.112729
min,0.010000,0.010000,0.010000,0.010000,0.000000,0.010000,0.000000,0.010000,0.010000,0.000000,0.000000,0.000000,8.000000
25%,26.200000,52.380000,3.840000,10.810000,10.660000,8.120000,0.420000,4.880000,13.420000,0.050000,0.370000,0.100000,79.000000
50%,46.420000,91.500000,7.960000,20.320000,20.790000,15.380000,0.800000,8.370000,26.240000,0.860000,2.590000,0.790000,116.000000
75%,79.490000,147.520000,16.150000,36.350000,37.150000,29.230000,1.370000,14.780000,47.620000,2.750000,8.410000,3.120000,208.000000
max,999.990000,1000.000000,499.990000,499.510000,498.610000,499.970000,498.570000,199.960000,497.620000,498.070000,499.400000,499.990000,3133.000000


In [142]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 707875 entries, 0 to 707874
Data columns (total 16 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   City        707875 non-null  str    
 1   Datetime    707875 non-null  str    
 2   PM2.5       562787 non-null  float64
 3   PM10        411138 non-null  float64
 4   NO          591243 non-null  float64
 5   NO2         590753 non-null  float64
 6   NOx         584651 non-null  float64
 7   NH3         435333 non-null  float64
 8   CO          621358 non-null  float64
 9   SO2         577502 non-null  float64
 10  O3          578667 non-null  float64
 11  Benzene     544229 non-null  float64
 12  Toluene     487268 non-null  float64
 13  Xylene      252046 non-null  float64
 14  AQI         578795 non-null  float64
 15  AQI_Bucket  578795 non-null  str    
dtypes: float64(13), str(3)
memory usage: 86.4 MB


In [143]:
df.isnull().sum()

City               0
Datetime           0
PM2.5         145088
PM10          296737
NO            116632
NO2           117122
NOx           123224
NH3           272542
CO             86517
SO2           130373
O3            129208
Benzene       163646
Toluene       220607
Xylene        455829
AQI           129080
AQI_Bucket    129080
dtype: int64

In [144]:
df.dropna(how="any",inplace=True)

In [145]:
df.describe()

,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI
count,129277.000000,129277.000000,129277.000000,129277.000000,129277.000000,129277.000000,129277.000000,129277.000000,129277.000000,129277.000000,129277.000000,129277.000000,129277.000000
mean,63.920899,128.279759,18.108749,32.815853,34.225690,21.820122,1.030950,11.281448,36.352084,3.912114,11.344988,2.566608,145.405873
std,62.152594,99.378185,28.985743,23.036079,35.307825,18.102877,1.572615,10.353600,27.687567,5.858445,15.042199,5.346189,97.011021
min,0.030000,1.000000,0.010000,0.100000,0.000000,0.010000,0.000000,0.010000,0.020000,0.000000,0.000000,0.000000,18.000000
25%,27.000000,63.750000,3.560000,15.420000,13.410000,10.320000,0.430000,5.500000,16.770000,0.940000,2.350000,0.220000,79.000000
50%,47.500000,101.500000,8.240000,27.200000,22.970000,15.350000,0.700000,8.830000,28.220000,2.250000,6.200000,1.000000,114.000000
75%,76.500000,157.400000,17.820000,44.100000,40.180000,29.560000,1.070000,13.600000,48.600000,4.560000,13.940000,2.690000,176.000000
max,999.990000,1000.000000,419.780000,266.730000,408.250000,485.520000,31.620000,199.930000,219.780000,120.080000,198.050000,261.950000,760.000000


In [146]:
df.info()

<class 'pandas.DataFrame'>
Index: 129277 entries, 50888 to 707867
Data columns (total 16 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   City        129277 non-null  str    
 1   Datetime    129277 non-null  str    
 2   PM2.5       129277 non-null  float64
 3   PM10        129277 non-null  float64
 4   NO          129277 non-null  float64
 5   NO2         129277 non-null  float64
 6   NOx         129277 non-null  float64
 7   NH3         129277 non-null  float64
 8   CO          129277 non-null  float64
 9   SO2         129277 non-null  float64
 10  O3          129277 non-null  float64
 11  Benzene     129277 non-null  float64
 12  Toluene     129277 non-null  float64
 13  Xylene      129277 non-null  float64
 14  AQI         129277 non-null  float64
 15  AQI_Bucket  129277 non-null  str    
dtypes: float64(13), str(3)
memory usage: 16.8 MB


In [147]:
df["City"].unique()

<StringArray>
[    'Amaravati',      'Amritsar',    'Chandigarh',         'Delhi',
      'Gurugram',     'Hyderabad',       'Kolkata',         'Patna',
 'Visakhapatnam']
Length: 9, dtype: str

In [148]:
df['Datetime'].unique()

<StringArray>
['2017-11-25 09:00:00', '2017-11-25 10:00:00', '2017-11-25 11:00:00',
 '2017-11-25 14:00:00', '2017-11-25 15:00:00', '2017-11-25 16:00:00',
 '2017-11-25 17:00:00', '2017-11-25 18:00:00', '2017-11-25 19:00:00',
 '2017-11-25 20:00:00',
 ...
 '2017-03-20 17:00:00', '2017-03-20 18:00:00', '2017-03-22 14:00:00',
 '2017-03-22 19:00:00', '2017-03-29 12:00:00', '2017-04-02 12:00:00',
 '2017-04-08 13:00:00', '2017-04-08 14:00:00', '2017-04-09 12:00:00',
 '2017-04-11 01:00:00']
Length: 42646, dtype: str

In [149]:
a=pd.get_dummies(df['City'],drop_first=True)

In [150]:
a.describe()

,Amritsar,Chandigarh,Delhi,Gurugram,Hyderabad,Kolkata,Patna,Visakhapatnam
count,129277,129277,129277,129277,129277,129277,129277,129277
unique,2,2,2,2,2,2,2,2
top,False,False,False,False,False,False,False,False
freq,116933,123783,100218,126555,92125,119925,124747,107325


In [151]:
df=df.drop('Datetime',axis=1)

In [152]:
df=pd.get_dummies(data=df,columns=['City'])

In [153]:
df.describe()

,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI
count,129277.000000,129277.000000,129277.000000,129277.000000,129277.000000,129277.000000,129277.000000,129277.000000,129277.000000,129277.000000,129277.000000,129277.000000,129277.000000
mean,63.920899,128.279759,18.108749,32.815853,34.225690,21.820122,1.030950,11.281448,36.352084,3.912114,11.344988,2.566608,145.405873
std,62.152594,99.378185,28.985743,23.036079,35.307825,18.102877,1.572615,10.353600,27.687567,5.858445,15.042199,5.346189,97.011021
min,0.030000,1.000000,0.010000,0.100000,0.000000,0.010000,0.000000,0.010000,0.020000,0.000000,0.000000,0.000000,18.000000
25%,27.000000,63.750000,3.560000,15.420000,13.410000,10.320000,0.430000,5.500000,16.770000,0.940000,2.350000,0.220000,79.000000
50%,47.500000,101.500000,8.240000,27.200000,22.970000,15.350000,0.700000,8.830000,28.220000,2.250000,6.200000,1.000000,114.000000
75%,76.500000,157.400000,17.820000,44.100000,40.180000,29.560000,1.070000,13.600000,48.600000,4.560000,13.940000,2.690000,176.000000
max,999.990000,1000.000000,419.780000,266.730000,408.250000,485.520000,31.620000,199.930000,219.780000,120.080000,198.050000,261.950000,760.000000


In [154]:
# in this we can see that AQI is highly correlated with PM 2.5 and PM 10

In [155]:
df.info()

<class 'pandas.DataFrame'>
Index: 129277 entries, 50888 to 707867
Data columns (total 23 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   PM2.5               129277 non-null  float64
 1   PM10                129277 non-null  float64
 2   NO                  129277 non-null  float64
 3   NO2                 129277 non-null  float64
 4   NOx                 129277 non-null  float64
 5   NH3                 129277 non-null  float64
 6   CO                  129277 non-null  float64
 7   SO2                 129277 non-null  float64
 8   O3                  129277 non-null  float64
 9   Benzene             129277 non-null  float64
 10  Toluene             129277 non-null  float64
 11  Xylene              129277 non-null  float64
 12  AQI                 129277 non-null  float64
 13  AQI_Bucket          129277 non-null  str    
 14  City_Amaravati      129277 non-null  bool   
 15  City_Amritsar       129277 non-null  bool   
 

In [156]:
#x=df.drop(["AQI_Bucket","AQI","City_Amaravati","City_Amritsar","City_Chandigarh","City_Delhi","City_Gurugram","City_Hyderabad","City_Kolkata","City_Patna","NOx","NH3","CO","SO2","O3","Benzene","Toluene","Xylene"],axis=1)

In [157]:
#y=df["AQI_Bucket"]

In [158]:
#model=LogisticRegression()

In [159]:
#model.fit(x_train,y_train)

In [160]:
#predictions=model.predict(x_test)

In [161]:
#print(confusion_matrix(y_test, predictions))
#print(accuracy_score(y_test, predictions))

In [162]:
x=df.drop(['AQI_Bucket'],axis=1)
x.head()

,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,...,AQI,City_Amaravati,City_Amritsar,City_Chandigarh,City_Delhi,City_Gurugram,City_Hyderabad,City_Kolkata,City_Patna,City_Visakhapatnam
50888,104.00,148.50,1.93,23.00,13.75,9.80,0.1,15.30,117.62,0.30,...,155.0,True,False,False,False,False,False,False,False,False
50889,94.50,142.00,1.33,16.25,9.75,9.65,0.1,17.00,136.23,0.28,...,159.0,True,False,False,False,False,False,False,False,False
50890,82.75,126.50,1.47,14.83,9.07,9.70,0.1,15.40,149.92,0.20,...,173.0,True,False,False,False,False,False,False,False,False
50893,68.50,117.00,1.35,13.60,8.35,7.40,0.1,21.80,161.70,0.10,...,191.0,True,False,False,False,False,False,False,False,False
50894,69.25,112.25,1.52,11.80,7.55,9.25,0.1,21.38,161.68,0.10,...,191.0,True,False,False,False,False,False,False,False,False


In [163]:
y=df.iloc[:,13]
y.head()
y.unique()

<StringArray>
['Moderate', 'Poor', 'Very Poor', 'Satisfactory', 'Good', 'Severe']
Length: 6, dtype: str

In [164]:
mp = {
    'Good': 0,
    'Satisfactory': 1,
    'Moderate': 2,
    'Poor': 3,
    'Very Poor': 4,
    'Severe': 5
}

y = y.map(mp)

In [165]:
params={
 "learning_rate"    : [0.05, 0.10, 0.15, 0.20, 0.25, 0.30 ] ,
 "max_depth"        : [ 3, 4, 5, 6, 8, 10, 12, 15],
 "min_child_weight" : [ 1, 3, 5, 7 ],
 "gamma"            : [ 0.0, 0.1, 0.2 , 0.3, 0.4 ],
 "colsample_bytree" : [ 0.3, 0.4, 0.5 , 0.7 ]  
}

In [166]:
grid_size_per_parameter  = [len(i) for i in params.values()]
print(grid_size_per_parameter)
np.prod(grid_size_per_parameter)

[6, 8, 4, 5, 4]


np.int64(3840)

In [167]:
classifier =xgb.XGBClassifier(device="cuda")

In [168]:
random_search=RandomizedSearchCV(classifier,
                                 param_distributions=params,
                                 n_iter=5,
                                 scoring='accuracy',
                                 n_jobs=-1,
                                 cv=5,
                                 verbose=3)

In [169]:
df.head()

,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,...,AQI_Bucket,City_Amaravati,City_Amritsar,City_Chandigarh,City_Delhi,City_Gurugram,City_Hyderabad,City_Kolkata,City_Patna,City_Visakhapatnam
50888,104.00,148.50,1.93,23.00,13.75,9.80,0.1,15.30,117.62,0.30,...,Moderate,True,False,False,False,False,False,False,False,False
50889,94.50,142.00,1.33,16.25,9.75,9.65,0.1,17.00,136.23,0.28,...,Moderate,True,False,False,False,False,False,False,False,False
50890,82.75,126.50,1.47,14.83,9.07,9.70,0.1,15.40,149.92,0.20,...,Moderate,True,False,False,False,False,False,False,False,False
50893,68.50,117.00,1.35,13.60,8.35,7.40,0.1,21.80,161.70,0.10,...,Moderate,True,False,False,False,False,False,False,False,False
50894,69.25,112.25,1.52,11.80,7.55,9.25,0.1,21.38,161.68,0.10,...,Moderate,True,False,False,False,False,False,False,False,False


In [170]:
type(random_search)

sklearn.model_selection._search.RandomizedSearchCV

In [171]:
#df.info()

In [172]:
random_search.estimator

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",'cuda'
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [173]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.33,random_state=1)

In [174]:
random_search.fit(x_train,y_train)

Fitting 5 folds for each of 5 candidates, totalling 25 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.3, 0.4, ...], 'gamma': [0.0, 0.1, ...], 'learning_rate': [0.05, 0.1, ...], 'max_depth': [3, 4, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",5
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",3
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits u

In [175]:
random_search.best_estimator_

,"objective objective: str | xgboost.objective.Objective | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.5
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",'cuda'
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True)

In [176]:
random_search.best_params_

{'min_child_weight': 1,
 'max_depth': 4,
 'learning_rate': 0.15,
 'gamma': 0.3,
 'colsample_bytree': 0.5}

In [183]:
from sklearn.metrics import accuracy_score

best_model = random_search.best_estimator_

y_pred = best_model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)

print("Test Accuracy:", accuracy)

Test Accuracy: 0.9988045567483943


In [177]:
classifier=xgb.XGBClassifier(base_score=0.5, booster=None, colsample_bylevel=1,
              colsample_bynode=1, colsample_bytree=0.5, gamma=0.2,
              importance_type='gain', interaction_constraints=None,
              learning_rate=0.05, max_delta_step=0, max_depth=10,
              min_child_weight=1, missing=None, monotone_constraints=None,
              n_estimators=100, n_jobs=0, num_parallel_tree=1,
              objective='multi:softprob', random_state=0, reg_alpha=0,
              reg_lambda=1, scale_pos_weight=None, subsample=1,
              tree_method=None, validate_parameters=False, verbosity=None)

In [184]:
from sklearn.model_selection import cross_val_score
score=cross_val_score(classifier,x_test,y_test,cv=10)

e:\code\AI study\AI study\ML\venv\Lib\site-packages\sklearn\model_selection\_validation.py:974: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "e:\code\AI study\AI study\ML\venv\Lib\site-packages\sklearn\metrics\_scorer.py", line 171, in __call__
    score = scorer(estimator, *args, **routed_params.get(name).score)
  File "e:\code\AI study\AI study\ML\venv\Lib\site-packages\sklearn\metrics\_scorer.py", line 531, in __call__
    return estimator.score(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "e:\code\AI study\AI study\ML\venv\Lib\site-packages\sklearn\base.py", line 628, in score
    return accuracy_score(y, self.predict(X), sample_weight=sample_weight)
                             ~~~~~~~~~~~~^^^
  File "e:\code\AI study\AI study\ML\venv\Lib\site-packages\xgboost\core.py", line 569, in inner_f
    return func(**kwargs)
  File "e:\code\AI study\AI 

In [179]:
score

array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan])

In [180]:
score.mean()*100

np.float64(nan)

In [181]:
score.std()*100

np.float64(nan)

**I'm just a Beginner
It's just for testing XGBClassifier algorithm, to analyse how it works;**
# recommendation are most welcomed
